In [ ]:
# ✈️ Flight Fare Prediction Using Machine Learning

This notebook uses the Excel files `datasets/Data_Train.xlsx` and `datasets/Test_set.xlsx` to build an end-to-end flight fare prediction workflow.
</VSCode.Cell>
<VSCode.Cell language="python">
import pandas as pd
import os

TRAIN_PATH = 'datasets/Data_Train.xlsx'
TEST_PATH = 'datasets/Test_set.xlsx'

train_df = pd.read_excel(TRAIN_PATH)
test_df = pd.read_excel(TEST_PATH)

print('TRAIN shape:', train_df.shape)
print('TEST shape:', test_df.shape)
print('\nTRAIN columns:', list(train_df.columns))
print('TEST columns:', list(test_df.columns))
print('\nTRAIN sample:')
display(train_df.head(3))
print('\nTEST sample:')
display(test_df.head(3))
</VSCode.Cell>
<VSCode.Cell language="python">
# Basic cleaning and validation
print('Missing values in train:')
print(train_df.isnull().sum())
print('\nMissing values in test:')
print(test_df.isnull().sum())

# Normalize common whitespace issues
for col in ['Airline', 'Source', 'Destination', 'Route', 'Additional_Info']:
    if col in train_df.columns:
        train_df[col] = train_df[col].astype(str).str.strip()
    if col in test_df.columns:
        test_df[col] = test_df[col].astype(str).str.strip()

# Keep a copy for later use
train_df_clean = train_df.copy()
test_df_clean = test_df.copy()

print('Train ready for preprocessing:', train_df_clean.shape)
print('Test ready for preprocessing:', test_df_clean.shape)
</VSCode.Cell>
<VSCode.Cell language="python">
import numpy as np

# Convert date and time fields into useful numeric features
train_df_clean['Date_of_Journey'] = pd.to_datetime(train_df_clean['Date_of_Journey'], errors='coerce')
train_df_clean['Dep_Time'] = pd.to_datetime(train_df_clean['Dep_Time'], format='mixed', errors='coerce')
train_df_clean['Arrival_Time'] = pd.to_datetime(train_df_clean['Arrival_Time'], format='mixed', errors='coerce')

# Duration parsing

def parse_duration(value):
    if pd.isna(value):
        return np.nan
    text = str(value).replace(' ', '')
    if 'h' in text and 'm' in text:
        h, m = text.split('h')
        m = m.replace('m', '')
        return int(h) * 60 + int(m)
    if 'h' in text:
        return int(text.replace('h', '').replace('m', '')) * 60
    if 'm' in text:
        return int(text.replace('m', ''))
    return np.nan

train_df_clean['Duration_Minutes'] = train_df_clean['Duration'].apply(parse_duration)

print(train_df_clean[['Date_of_Journey', 'Dep_Time', 'Arrival_Time', 'Duration', 'Duration_Minutes']].head())
</VSCode.Cell>
<VSCode.Cell language="python">
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
sns.histplot(train_df_clean['Price'], bins=30, kde=True)
plt.title('Price Distribution')
plt.subplot(1, 2, 2)
sns.boxplot(y=train_df_clean['Price'])
plt.title('Price Spread')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.countplot(y='Airline', data=train_df_clean, order=train_df_clean['Airline'].value_counts().index)
plt.title('Flights by Airline')
plt.show()
</VSCode.Cell>
<VSCode.Cell language="python">
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Prepare features and target
feature_cols = ['Airline', 'Source', 'Destination', 'Total_Stops', 'Dep_Time', 'Arrival_Time', 'Duration_Minutes']
X = train_df_clean[feature_cols].copy()
y = train_df_clean['Price']

# Simple feature engineering from time fields
X['Dep_Hour'] = X['Dep_Time'].dt.hour
X['Arr_Hour'] = X['Arrival_Time'].dt.hour
X['Journey_Month'] = X['Date_of_Journey'].dt.month
X['Journey_Weekday'] = X['Date_of_Journey'].dt.dayofweek

# Build categorical and numeric transformers
num_cols = ['Duration_Minutes', 'Dep_Hour', 'Arr_Hour', 'Journey_Month', 'Journey_Weekday']
cat_cols = ['Airline', 'Source', 'Destination', 'Total_Stops']

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols)
])

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_val)
    mae = mean_absolute_error(y_val, pred)
    rmse = mean_squared_error(y_val, pred, squared=False)
    r2 = r2_score(y_val, pred)
    print(f'[{name}] MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.3f}')
</VSCode.Cell>
<VSCode.Cell language="python">
# Fit the best model on the full training set (Random Forest here)
final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=300, random_state=42))
])
final_model.fit(X, y)

print('Training complete for the selected model.')
</VSCode.Cell>
<VSCode.Cell language="python">
# Save the trained model and a small summary for reuse
import joblib
import json

os.makedirs('models', exist_ok=True)
joblib.dump(final_model, 'models/flight_fare_model.pkl')

metrics = {
    'model': 'RandomForestRegressor',
    'train_shape': list(train_df_clean.shape),
    'feature_columns': list(X.columns),
    'target': 'Price'
}
with open('models/flight_fare_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print('Saved model to models/flight_fare_model.pkl')
print('Saved metrics to models/flight_fare_metrics.json')
</VSCode.Cell>
